# Tableau → Fabric: VizQL Data Service Bridge — Play 3

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook is the **data loading** stage of the pipeline. It reads the datasource
and field inventory produced by Play 3 (Tableau Metadata Bridge), then uses the
VizQL Data Service (VDS) REST API to pull each upstream table from each datasource
into the Fabric Lakehouse as individual Delta tables.

**Pipeline order:** Play 2 → Play 3 → Play 4

```
Metadata_Lakehouse (Play 2 output)
  tableau_datasources  ← which datasources exist
  tableau_fields       ← which fields belong to which upstream table
        ↓
Play 3 (this notebook)
  For each datasource × upstream table:
    VDS query (fields for that table only → no joins)
        ↓
h1_ultrastore Lakehouse
  {datasource_name}_{table_name}  ← one Delta table per upstream table
        ↓
Play 4 → semantic model generation
```

**Delta table naming convention:** `{datasource_name}_{table_name}`
e.g. `superstore_datasource_orders`, `superstore_datasource_people`

---

**Prerequisites**
- Play 2 has been run and Metadata_Lakehouse tables are current
- Tableau Cloud or Tableau Server 2025.1+
- Creator license on the Tableau site
- Personal Access Token (PAT) stored in Azure Key Vault
- h1_ultrastore Lakehouse attached to this notebook

**Cells in this notebook**
1. Configuration
2. Authenticate to Tableau
3. Load Play 2 metadata
4. Main loop — query VDS per table and write Delta tables
5. Verification


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `PAT_NAME` | Tableau PAT name | Tableau → Account Settings → Personal Access Tokens |
| `POD` | Tableau Cloud pod hostname | First part of your Tableau Cloud URL |
| `SITE` | Site contentUrl slug | Your site URL slug. Use `""` for Tableau Server default site |
| `KV_URL` | Azure Key Vault URL | portal.azure.com → your Key Vault → Overview → Vault URI |
| `KV_SECRET_NAME` | PAT secret name in Key Vault | The secret name you used when storing the PAT |
| `METADATA_LAKEHOUSE` | Name of the Play 2 metadata lakehouse | e.g. `Metadata_Lakehouse` |
| `VDS_RATE_LIMIT` | Max VDS calls per hour | 100 × number of Creator licenses on your Tableau site |
| `DATASOURCE_FILTER` | Optional list of datasource names to process | Leave empty `[]` to process all |
| `BATCH_SIZE` | Max datasources per run | Use with `BATCH_OFFSET` to chunk large deployments |
| `BATCH_OFFSET` | Starting position in datasource list | Increment by `BATCH_SIZE` for next chunk |


## Cell 1 — Configuration

Set your Tableau environment details and pipeline controls here.
The PAT secret is retrieved securely from Azure Key Vault.

> 🔄 **Adapting for your environment:** Update `POD`, `SITE`, `KV_URL`, `KV_SECRET_NAME`,
> and `METADATA_LAKEHOUSE`. Adjust `VDS_RATE_LIMIT` based on your Creator license count.
> Use `DATASOURCE_FILTER` and batch controls for large deployments.

In [1]:
# ── TABLEAU CONNECTION ────────────────────────────────────────────────────────
PAT_NAME         = ""                                    # PAT name from Tableau account settings
POD              = ""                                    # e.g. 10ay.online.tableau.com
SITE             = ""                                    # Site contentUrl slug. Use "" for default site

KV_URL           = "https://<your-keyvault-name>.vault.azure.net/"
KV_SECRET_NAME   = "<your-secret-name>"

# ── LAKEHOUSE SETTINGS ───────────────────────────────────────────────────────
METADATA_LAKEHOUSE = "Metadata_Lakehouse"                # Play 2 output lakehouse name
# Note: h1_ultrastore (data lakehouse) must be attached as default lakehouse

# ── RATE LIMITING ────────────────────────────────────────────────────────────
VDS_RATE_LIMIT   = 100                                   # VDS calls/hour (100 × Creator license count)

# ── BATCH / FILTER CONTROLS ──────────────────────────────────────────────────
DATASOURCE_FILTER = []   # e.g. ["Superstore Datasource", "World Statistics"]
                         # Leave empty to process all datasources
PROJECT_FILTER    = []   # e.g. ["Finance", "Marketing"]
                         # Leave empty to process all projects
BATCH_SIZE        = 0    # Max datasources per run. 0 = no limit
BATCH_OFFSET      = 0    # Starting position. Increment by BATCH_SIZE for next chunk

# ── SECURE CREDENTIAL RETRIEVAL ──────────────────────────────────────────────
PAT_SECRET = notebookutils.credentials.getSecret(KV_URL, KV_SECRET_NAME)

BASE = f"https://{POD}"

import math
RATE_LIMIT_DELAY = 3600 / VDS_RATE_LIMIT   # seconds between VDS calls

print("✓ Configuration loaded")
print(f"  Pod:                {POD}")
print(f"  Site:               {SITE}")
print(f"  Metadata lakehouse: {METADATA_LAKEHOUSE}")
print(f"  VDS rate limit:     {VDS_RATE_LIMIT} calls/hour ({RATE_LIMIT_DELAY:.1f}s delay)")
print(f"  Datasource filter:  {DATASOURCE_FILTER or 'all'}")
print(f"  Project filter:     {PROJECT_FILTER or 'all'}")
print(f"  Batch:              offset={BATCH_OFFSET}, size={BATCH_SIZE or 'unlimited'}")
print(f"  PAT secret:         retrieved from Key Vault ✓")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 3, Finished, Available, Finished, False)

✓ Configuration loaded
  Pod:                10ay.online.tableau.com
  Site:               vdsapi82-5507b0d30c
  Metadata lakehouse: Metadata_Lakehouse
  VDS rate limit:     100 calls/hour (36.0s delay)
  Datasource filter:  all
  Project filter:     all
  Batch:              offset=0, size=unlimited
  PAT secret:         retrieved from Key Vault ✓


## Cell 2 — Authenticate to Tableau

Authenticates using your PAT and retrieves a session token.

> **If you get a 401 error in later cells**, re-run this cell to refresh the session token.
> Tokens expire after inactivity or if another session opens with the same PAT.

In [2]:
import requests
import json
import pandas as pd
import time
import re
from datetime import datetime

auth_response = requests.post(
    f"{BASE}/api/3.24/auth/signin",
    json={
        "credentials": {
            "personalAccessTokenName": PAT_NAME,
            "personalAccessTokenSecret": PAT_SECRET,
            "site": {"contentUrl": SITE}
        }
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"}
)
auth_response.raise_for_status()

auth_data = auth_response.json()
TOKEN   = auth_data["credentials"]["token"]
SITE_ID = auth_data["credentials"]["site"]["id"]

HEADERS = {
    "X-Tableau-Auth": TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print("✓ Authenticated to Tableau")
print(f"  Token:    {TOKEN[:8]}...")
print(f"  Site ID:  {SITE_ID}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 4, Finished, Available, Finished, False)

✓ Authenticated to Tableau
  Token:    esSGFbHj...
  Site ID:  8c95a2ed-1221-46a6-ae25-1ff1a34e4c80


## Cell 3 — Load Play 3 Metadata

Reads the datasource and field inventory from the Metadata_Lakehouse tables
produced by Play 3. This is the manifest that drives the entire data loading loop.

Applies any configured filters (datasource name, project, batch offset/size)
so large deployments can be processed in chunks.

In [3]:
from pyspark.sql.types import NullType

def read_metadata_table(table_name):
    """Read a table from the Metadata Lakehouse, safely dropping void columns."""
    df = spark.sql(f"SELECT * FROM {METADATA_LAKEHOUSE}.dbo.{table_name}")
    void_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NullType)]
    if void_cols:
        df = df.drop(*void_cols)
    return df

# Load datasources inventory from Play 2
df_datasources = read_metadata_table("tableau_datasources").toPandas()

# Apply filters
if DATASOURCE_FILTER:
    df_datasources = df_datasources[df_datasources["name"].isin(DATASOURCE_FILTER)]
if PROJECT_FILTER:
    df_datasources = df_datasources[df_datasources["project_name"].isin(PROJECT_FILTER)]

# Apply batch offset/size
df_datasources = df_datasources.reset_index(drop=True)
if BATCH_SIZE > 0:
    df_datasources = df_datasources.iloc[BATCH_OFFSET:BATCH_OFFSET + BATCH_SIZE]
else:
    df_datasources = df_datasources.iloc[BATCH_OFFSET:]

# Load fields from Play 2 — ColumnFields with known source_table only
# These are the field captions passed to VDS queries
df_fields = read_metadata_table("tableau_fields").toPandas()
df_fields = df_fields[
    (df_fields["field_type"] == "ColumnField") &
    (df_fields["source_table"].notna())
]

print("✓ Metadata loaded from Play 2")
print(f"  Datasources to process: {len(df_datasources)}")
for _, ds in df_datasources.iterrows():
    tables = df_fields[df_fields["datasource_id"] == ds["datasource_id"]]["source_table"].unique()
    print(f"    • {ds['name']} → {list(tables)}")
print(f"  Total ColumnFields with source_table: {len(df_fields)}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 5, Finished, Available, Finished, False)

✓ Metadata loaded from Play 3
  Datasources to process: 1
    • Superstore Datasource → ['Orders', 'People', 'Returns']
  Total ColumnFields with source_table: 25


## Cell 4 — Main Loop: Query VDS and Write Delta Tables

For each datasource, for each upstream table:
1. Builds the field list from Play 2 metadata (only fields belonging to that table)
2. Queries VDS with those fields only — no cross-table joins, full row counts preserved
3. Sanitizes column names for Delta compatibility
4. Writes to `{datasource_name}_{table_name}` in the attached lakehouse
5. Respects the VDS rate limit with a configurable delay between calls

> **On failure:** each table write is independent. If one fails, the loop continues
> and logs the error. Re-run with `DATASOURCE_FILTER` to retry specific datasources.

> **Rate limiting:** `VDS_RATE_LIMIT` in Cell 1 controls the delay between calls.
> Set to `100 × number of Creator licenses` on your Tableau site.

In [4]:
def clean_col(name):
    """Replace Delta-incompatible characters with underscores."""
    for ch in ["(", ")", " ", ",", ";", "{", "}", "/", "\\", "\n", "\t", "="]:
        name = name.replace(ch, "_")
    return name.strip("_")

def make_table_name(datasource_name, table_name):
    """Generate Delta table name from datasource and upstream table names."""
    def slugify(s):
        s = s.lower().strip()
        s = re.sub(r'[^a-z0-9]+', '_', s)
        return s.strip('_')
    return f"{slugify(datasource_name)}_{slugify(table_name)}"

def get_datasource_luid(datasource_name):
    """Look up the LUID for a datasource by name via REST API."""
    resp = requests.get(
        f"{BASE}/api/3.24/sites/{SITE_ID}/datasources",
        headers=HEADERS
    )
    resp.raise_for_status()
    datasources = resp.json().get("datasources", {}).get("datasource", [])
    if isinstance(datasources, dict):
        datasources = [datasources]
    match = next((ds for ds in datasources if ds["name"].lower() == datasource_name.lower()), None)
    if not match:
        raise ValueError(f"Datasource '{datasource_name}' not found on site")
    return match["id"]

def get_logical_tables(luid):
    """
    Call VDS read-metadata and return unique logical table names.
    Uses logicalTableId prefix (before the GUID) as the table name.
    Only used for table discovery — not for field lists.
    """
    resp = requests.post(
        f"{BASE}/api/v1/vizql-data-service/read-metadata",
        json={"datasource": {"datasourceLuid": luid}},
        headers=HEADERS
    )
    resp.raise_for_status()
    fields = resp.json().get("data", [])
    tables = set()
    for f in fields:
        logical_table_id = f.get("logicalTableId", "")
        if logical_table_id:
            # Extract table name — format is TableName_<GUID>, strip last 5 underscore segments
            table_name = logical_table_id.rsplit("_", 5)[0]
            tables.add(table_name)
    return list(tables)

def query_vds_fields(luid, field_captions):
    """
    Query VDS for a list of field captions.
    If the batch query fails with 400, retries field by field,
    dropping any fields that VDS doesn't recognize.
    Returns (DataFrame, list of dropped fields).
    """
    query_fields = [{"fieldCaption": f} for f in field_captions]
    resp = requests.post(
        f"{BASE}/api/v1/vizql-data-service/query-datasource",
        json={
            "datasource": {"datasourceLuid": luid},
            "query": {"fields": query_fields},
            "options": {"returnFormat": "OBJECTS"}
        },
        headers=HEADERS
    )

    if resp.status_code == 200:
        rows = resp.json().get("data", [])
        return pd.DataFrame(rows), []

    if resp.status_code == 400:
        # Retry field by field — drop unrecognized fields
        print(f"    ⚠ Batch query failed, retrying field by field...")
        good_captions = []
        dropped = []
        for caption in field_captions:
            test = requests.post(
                f"{BASE}/api/v1/vizql-data-service/query-datasource",
                json={
                    "datasource": {"datasourceLuid": luid},
                    "query": {"fields": [{"fieldCaption": caption}]},
                    "options": {"returnFormat": "OBJECTS"}
                },
                headers=HEADERS
            )
            if test.status_code == 200:
                good_captions.append(caption)
            else:
                dropped.append(caption)
                print(f"    ✗ Dropped unrecognized field: {caption}")

        if not good_captions:
            raise ValueError("No queryable fields found for this table")

        # Final query with only good fields
        resp2 = requests.post(
            f"{BASE}/api/v1/vizql-data-service/query-datasource",
            json={
                "datasource": {"datasourceLuid": luid},
                "query": {"fields": [{"fieldCaption": f} for f in good_captions]},
                "options": {"returnFormat": "OBJECTS"}
            },
            headers=HEADERS
        )
        resp2.raise_for_status()
        rows = resp2.json().get("data", [])
        return pd.DataFrame(rows), dropped

    resp.raise_for_status()  # raise for any other error

def write_delta(df_pandas, table_name):
    """Write pandas DataFrame to Delta table in attached lakehouse."""
    df_spark = spark.createDataFrame(df_pandas)
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)
    return spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
results      = []
errors       = []
dropped_log  = []
vds_call_count = 0

print(f"Starting VDS data load — {datetime.utcnow().isoformat()}")
print("=" * 60)

for _, ds_row in df_datasources.iterrows():
    ds_id   = ds_row["datasource_id"]
    ds_name = ds_row["name"]

    print(f"\n── {ds_name} ──")

    # Get LUID
    try:
        luid = get_datasource_luid(ds_name)
        print(f"  LUID: {luid}")
    except Exception as e:
        print(f"  ✗ Could not resolve LUID: {e}")
        errors.append({"datasource": ds_name, "table": "N/A", "error": str(e)})
        continue

    # Get logical table names from VDS (table discovery only)
    try:
        logical_tables = get_logical_tables(luid)
        print(f"  Logical tables: {logical_tables}")
    except Exception as e:
        print(f"  ✗ Could not read VDS metadata: {e}")
        errors.append({"datasource": ds_name, "table": "N/A", "error": str(e)})
        continue

    # Get Play 3 fields for this datasource — source of truth for field captions
    ds_fields = df_fields[df_fields["datasource_id"] == ds_id]

    for table_name in logical_tables:
        delta_table = make_table_name(ds_name, table_name)

        # Get field captions from Play 2 metadata for this table
        table_fields = ds_fields[ds_fields["source_table"] == table_name]["field_name"].tolist()

        if not table_fields:
            print(f"  → {table_name}: no fields in Play 2 metadata, skipping")
            continue

        print(f"  → {table_name} ({len(table_fields)} fields) → {delta_table}")

        # Rate limit
        if vds_call_count > 0:
            time.sleep(RATE_LIMIT_DELAY)

        try:
            df, dropped = query_vds_fields(luid, table_fields)
            vds_call_count += 1

            if dropped:
                dropped_log.append({
                    "datasource": ds_name,
                    "table": table_name,
                    "dropped_fields": dropped
                })

            if len(df) == 0:
                print(f"    ⚠ Skipped — 0 rows returned")
                continue

            # Sanitize column names for Delta
            df.columns = [clean_col(c) for c in df.columns]

            row_count = write_delta(df, delta_table)
            print(f"    ✓ {row_count} rows written")
            results.append({
                "datasource": ds_name,
                "table": table_name,
                "delta_table": delta_table,
                "rows": row_count,
                "dropped_fields": len(dropped)
            })

        except Exception as e:
            print(f"    ✗ Failed: {e}")
            errors.append({"datasource": ds_name, "table": table_name, "error": str(e)})
            vds_call_count += 1

print(f"\n{'=' * 60}")
print(f"✓ Complete — {datetime.utcnow().isoformat()}")
print(f"  VDS calls made:   {vds_call_count}")
print(f"  Tables written:   {len(results)}")
print(f"  Errors:           {len(errors)}")
if dropped_log:
    print(f"\n  Dropped fields (VDS name mismatch):")
    for d in dropped_log:
        print(f"    {d['datasource']} / {d['table']}: {d['dropped_fields']}")
if errors:
    print(f"\n  Failed tables:")
    for e in errors:
        print(f"    ✗ {e['datasource']} / {e['table']}: {e['error']}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 6, Finished, Available, Finished, False)

Starting VDS data load — 2026-05-20T04:28:23.007916

── Superstore Datasource ──
  LUID: 0b2344cd-0347-400f-8107-e7ed8139abc3
  Logical tables: ['Orders', 'People', 'Returns']
  → Orders (21 fields) → superstore_datasource_orders
    ✓ 10194 rows written
  → People (2 fields) → superstore_datasource_people
    ⚠ Batch query failed, retrying field by field...
    ✗ Dropped unrecognized field: Person
    ✓ 4 rows written
  → Returns (2 fields) → superstore_datasource_returns
    ✓ 296 rows written

✓ Complete — 2026-05-20T04:30:05.962472
  VDS calls made:   3
  Tables written:   3
  Errors:           0

  Dropped fields (VDS name mismatch):
    Superstore Datasource / People: ['Person']


## Cell 5 — Verification

Confirms all expected Delta tables were written with the correct row counts.
Also shows the full list of tables written this run for handoff to Play 4.

In [5]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

if results:
    print("\n── Tables written this run ──")
    print(f"  {'Datasource':<30} {'Source Table':<20} {'Delta Table':<45} {'Rows'}")
    print(f"  {'-'*30} {'-'*20} {'-'*45} {'-'*8}")
    for r in results:
        print(f"  {r['datasource']:<30} {r['table']:<20} {r['delta_table']:<45} {r['rows']}")

    print(f"\n  Total tables: {len(results)}")
    print(f"  Total rows:   {sum(r['rows'] for r in results)}")
    print(f"\n  ✓ Delta tables ready for Play 4 semantic model generation")
    print(f"  ✓ Table naming: {{datasource_name}}_{{upstream_table}}")
else:
    print("  No tables written this run.")

if errors:
    print(f"\n── {len(errors)} error(s) — retry with DATASOURCE_FILTER ──")
    for e in errors:
        print(f"  ✗ {e['datasource']} / {e['table']}: {e['error']}")


StatementMeta(, d84f646d-c287-4db2-b91a-8f11acc01f6d, 7, Finished, Available, Finished, False)

VERIFICATION

── Tables written this run ──
  Datasource                     Source Table         Delta Table                                   Rows
  ------------------------------ -------------------- --------------------------------------------- --------
  Superstore Datasource          Orders               superstore_datasource_orders                  10194
  Superstore Datasource          People               superstore_datasource_people                  4
  Superstore Datasource          Returns              superstore_datasource_returns                 296

  Total tables: 3
  Total rows:   10494

  ✓ Delta tables ready for Play 4 semantic model generation
  ✓ Table naming: {datasource_name}_{upstream_table}
